<a href="https://colab.research.google.com/github/xxcorn888-cyber/solubility-prediction/blob/main/bbb_crossvalidation_and_checks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rdkit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 37.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd, numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

bbb = pd.read_csv("https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv")
bbb["mol"] = bbb["smiles"].apply(Chem.MolFromSmiles)
bbb = bbb[bbb["mol"].notnull()]

X_desc = np.array([[Descriptors.MolWt(m), Descriptors.MolLogP(m), Descriptors.TPSA(m),
                      Descriptors.NumHDonors(m), Descriptors.NumHAcceptors(m)]
                     for m in bbb["mol"]])
X_fp = np.array([[int(b) for b in AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048).ToBitString()]
                   for m in bbb["mol"]])
y = bbb["p_np"].values

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, Xi in [("描述符", X_desc), ("指纹", X_fp)]:
  s = cross_val_score(RandomForestClassifier(random_state=42), Xi, y,
                          cv=cv, scoring="roc_auc")
  print(name, np.round(s, 3), "平均", round(s.mean(), 3))

[16:02:24] Explicit valence for atom # 1 N, 4, is greater than permitted
[16:02:24] WARNING: not removing hydrogen atom without neighbors
[16:02:24] Explicit valence for atom # 6 N, 4, is greater than permitted
[16:02:24] WARNING: not removing hydrogen atom without neighbors
[16:02:24] WARNING: not removing hydrogen atom without neighbors
[16:02:24] WARNING: not removing hydrogen atom without neighbors
[16:02:24] WARNING: not removing hydrogen atom without neighbors
[16:02:24] WARNING: not removing hydrogen atom without neighbors
[16:02:24] WARNING: not removing hydrogen atom without neighbors
[16:02:24] Explicit valence for atom # 6 N, 4, is greater than permitted
[16:02:24] WARNING: not removing hydrogen atom without neighbors
[16:02:24] WARNING: not removing hydrogen atom without neighbors
[16:02:24] WARNING: not removing hydrogen atom without neighbors
[16:02:24] WARNING: not removing hydrogen atom without neighbors
[16:02:24] Explicit valence for atom # 11 N, 4, is greater than pe

描述符 [0.848 0.862 0.868 0.881 0.892] 平均 0.87
指纹 [0.915 0.908 0.922 0.913 0.917] 平均 0.915


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem

m = Chem.MolFromSmiles("C1=CC(=CC=C1C2=C(C(=O)C3=C(C=C(C=C3O2)O)O)O)O")  # 山奈酚

Xd_tr, Xd_te, y_tr, y_te = train_test_split(X_desc, y, test_size=0.2, random_state=42)
Xf_tr, Xf_te, _, _       = train_test_split(X_fp,  y, test_size=0.2, random_state=42)

clf_d = RandomForestClassifier(random_state=42).fit(Xd_tr, y_tr)
clf_f = RandomForestClassifier(random_state=42).fit(Xf_tr, y_tr)

d_feat = [[Descriptors.MolWt(m), Descriptors.MolLogP(m), Descriptors.TPSA(m),
             Descriptors.NumHDonors(m), Descriptors.NumHAcceptors(m)]]
f_feat = [[int(b) for b in AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048).ToBitString()]]

print("描述符版给山奈酚:", round(clf_d.predict_proba(d_feat)[0][1], 3))
print("指纹版给山奈酚:  ", round(clf_f.predict_proba(f_feat)[0][1], 3))
print("森林里有几棵树:  ", clf_f.n_estimators)

p = clf_f.predict_proba(Xf_te)[:, 1]
fpr, tpr, thr = roc_curve(y_te, p)
print("约登指数给的最佳阈值:", round(thr[np.argmax(tpr - fpr)], 3))

描述符版给山奈酚: 0.522
指纹版给山奈酚:   0.49
森林里有几棵树:   100
约登指数给的最佳阈值: 0.81


[16:03:28] DEPRECATION WARNING: please use MorganGenerator


In [ ]:
smi = "C1=CC(=CC=C1C2=C(C(=O)C3=C(C=C(C=C3O2)O)O)O)O"
m = Chem.AddHs(Chem.MolFromSmiles(smi))
AllChem.EmbedMultipleConfs(m, numConfs=10, randomSeed=42)
AllChem.MMFFOptimizeMoleculeConfs(m)
print(m.GetNumConformers())

10


In [ ]:
hs = []
ks = []
for a in m.GetAtoms():
      if a.GetSymbol() == "H":
          if a.GetNeighbors()[0].GetSymbol() == "O":
              hs.append(a.GetIdx())
      if a.GetSymbol() == "O":
          nh = 0
          for nb in a.GetNeighbors():
              if nb.GetSymbol() == "H":
                  nh = 1
          db = 0
          for b in a.GetBonds():
              if b.GetBondType() == Chem.BondType.DOUBLE:
                  db = 1
          if nh == 0 and db == 1:
              ks.append(a.GetIdx())
print(len(hs), len(ks))

4 1


In [ ]:
ds = []
for c in m.GetConformers():
      P = c.GetPositions()
      best = 99.0
      for h in hs:
          for o in ks:
              d = np.linalg.norm(P[h] - P[o])
              if d < best:
                  best = d
      ds.append(round(best, 2))
print(ds)

[np.float64(1.81), np.float64(1.81), np.float64(1.81), np.float64(1.81), np.float64(1.81), np.float64(1.81), np.float64(1.81), np.float64(1.81), np.float64(1.81), np.float64(1.81)]
